# 資料增強與遷移學習

對應原作業：Day 99–100。這是現代 PyTorch 練習；原始作業保留在 `../../homework/`。

從上到下執行即可。預設使用 CPU、合成資料，不需下載外部資料；改為真實資料時，先決定資料粒度與切分方式。

## 何時用
少量影像分類優先試預訓練模型；隨機增強只放訓練集。此 notebook 預設 `weights=None` 以便離線檢查架構；改成 `ResNet18_Weights.DEFAULT` 才是真正的預訓練，首次使用會下載權重。請依官方安裝選擇器安裝相容的 torch／torchvision。

In [ ]:
import numpy as np
import torch
from PIL import Image
from torch import nn
from torchvision.transforms import v2
from torchvision.models import ResNet18_Weights, resnet18

image = Image.fromarray(np.full((224, 224, 3), 120, dtype=np.uint8))
train_transform = v2.Compose([v2.RandomHorizontalFlip(p=0.5), v2.ToImage(),
                              v2.ToDtype(torch.float32, scale=True)])
train_tensor = train_transform(image)
assert train_tensor.shape == (3, 224, 224)
print("augmented tensor:", train_tensor.shape)


In [ ]:
USE_PRETRAINED = False  # 若改 True，需連網／已快取權重
weights = ResNet18_Weights.DEFAULT if USE_PRETRAINED else None
model = resnet18(weights=weights)
for parameter in model.parameters():
    parameter.requires_grad = False
model.fc = nn.Linear(model.fc.in_features, 2)
model.eval()
with torch.inference_mode():
    output = model(train_tensor.unsqueeze(0))
assert output.shape == (1, 2)
print("output shape:", output.shape)

if weights is not None:
    validation_transform = weights.transforms()
    print("pretrained weights need their own validation_transform:", validation_transform)
